# List Comprehensions & Generators

## Learning Map

`List comprehension → iterable comprehension → nested comprehension → conditional comprehension → dict comprehension → generator expression → lazy transformation → custom generator → time-series filtering`

## 1. Write a basic list comprehension

**Concept:** A list comprehension creates a new list by transforming every element of an iterable.

In [ ]:
class PriceSeries:
    def __init__(self, ticker, prices):
        self.ticker, self.prices = ticker, prices

    def adjusted_prices(self, uplift):
        return [price * (1 + uplift) for price in self.prices]

series = PriceSeries("BHP", [41.70, 42.15, 41.92, 43.08, 42.76])
print(series.adjusted_prices(0.01))

## 2. List comprehension over iterables

**Concept:** A comprehension can consume any iterable, including a custom iterable class.

In [ ]:
class Portfolio:
    def __init__(self, positions): 
        self.positions = positions
    def __iter__(self): 
        return iter(self.positions.items())

portfolio = Portfolio({"BHP": .18, "CBA": .27, "CSL": .20, "RIO": .20, "WES": .15})
weights = [weight for ticker, weight in portfolio]
large = [ticker for ticker, weight in portfolio if weight >= .20]
print(weights)
print(large)

[0.18, 0.27, 0.2, 0.2, 0.15]
['CBA', 'CSL', 'RIO']


## 3. Writing list comprehensions

**Concept:** The expression can calculate a business metric rather than simply copying the source value.

In [ ]:
class TradeBook:
    def __init__(self, trades):
        self.trades = trades
    def notionals(self):
        return [t["quantity"] * t["price"] for t in self.trades]
    def buy_notionals(self):
        return [t["quantity"] * t["price"] for t in self.trades if t["side"] == "BUY"]

book = TradeBook([
    {"ticker":"BHP","side":"BUY","quantity":2400,"price":41.70},
    {"ticker":"CBA","side":"SELL","quantity":800,"price":178.25},
    {"ticker":"CSL","side":"BUY","quantity":350,"price":312.40},
    {"ticker":"RIO","side":"BUY","quantity":1200,"price":119.80},
])
print(book.notionals())
print(book.buy_notionals())

[100080.0, 142600.0, 109339.99999999999, 143760.0]
[100080.0, 109339.99999999999, 143760.0]


## 4. Nested list comprehensions

**Concept:** Nested comprehensions express multiple iteration levels and are useful for matrix or multi-day market data.

In [ ]:
class MarketMatrix:
    def __init__(self, prices): 
        self.prices = prices
    def flatten(self): 
        return [price for day in self.prices for price in day]
    def returns(self):
        return [[round(row[i]/row[i-1]-1,4) for i in range(1,len(row))] for row in self.prices]

matrix = MarketMatrix([[41.70,42.15,41.92,43.08],[178.25,179.10,177.80,180.20],[312.40,315.10,313.70,318.20]])
print(matrix.flatten())
print(matrix.returns())

[41.7, 42.15, 41.92, 43.08, 178.25, 179.1, 177.8, 180.2, 312.4, 315.1, 313.7, 318.2]
[[0.0108, -0.0055, 0.0277], [0.0048, -0.0073, 0.0135], [0.0086, -0.0044, 0.0143]]


## 5. Advanced comprehensions

**Concept:** Advanced comprehensions can combine filtering, transformation, and conditional output in one expression.

In [ ]:
class RiskScanner:
    def __init__(self, observations): 
        self.observations = observations
    def report(self):
        return [{"ticker":r["ticker"], "volatility":r["volatility"], "risk": "HIGH" if r["volatility"]>=.30 else "MEDIUM" if r["volatility"]>=.20 else "LOW"}
                for r in self.observations if r["volume"] >= 500_000]

scanner = RiskScanner([
    {"ticker":"BHP","volatility":.28,"volume":1842000},
    {"ticker":"CBA","volatility":.19,"volume":932100},
    {"ticker":"CSL","volatility":.31,"volume":521800},
    {"ticker":"RIO","volatility":.34,"volume":1205000},
])
for row in scanner.report(): 
    print(row)

{'ticker': 'BHP', 'volatility': 0.28, 'risk': 'MEDIUM'}
{'ticker': 'CBA', 'volatility': 0.19, 'risk': 'LOW'}
{'ticker': 'CSL', 'volatility': 0.31, 'risk': 'HIGH'}
{'ticker': 'RIO', 'volatility': 0.34, 'risk': 'HIGH'}


## 6. Using conditionals in comprehensions (1)

**Concept:** A trailing `if` filters elements before they enter the output list.

In [3]:
class MarketFilter:
    def __init__(self, records):
        self.records = records
    def high_volume(self, threshold):
        return [r["ticker"] for r in self.records if r["volume"] >= threshold]

records = [{"ticker":"BHP","volume":1842000},{"ticker":"CBA","volume":932100},{"ticker":"CSL","volume":521800},{"ticker":"WES","volume":310000}]
print(MarketFilter(records).high_volume(800000))

['BHP', 'CBA']


## 7. Using conditionals in comprehensions (2)

**Concept:** A conditional expression can change the output value while a separate `if` filters records.

In [ ]:
class ReturnClassifier:
    def __init__(self, records): self.records = records
    def labels(self):
        return [f"{r['ticker']}: {'GAIN' if r['return'] > 0 else 'LOSS'}"
                for r in self.records if abs(r['return']) >= .01]

records = [{"ticker":"BHP","return":.021},{"ticker":"CBA","return":-.008},{"ticker":"CSL","return":.014},{"ticker":"RIO","return":-.023},{"ticker":"WES","return":.004}]
for label in ReturnClassifier(records).labels(): print(label)

## 8. Dict comprehensions

**Concept:** A dictionary comprehension constructs key-value mappings from an iterable.

In [4]:
class PortfolioAnalytics:
    def __init__(self, positions):
        self.positions = positions
    def exposure(self, value):
        return {ticker: weight*value for ticker,weight in self.positions.items()}
    def material(self, minimum):
        return {ticker:weight for ticker,weight in self.positions.items() if weight >= minimum}

p = PortfolioAnalytics({"BHP":.18,"CBA":.27,"CSL":.20,"RIO":.20,"WES":.15})
print(p.exposure(5000000))
print(p.material(0.20))

{'BHP': 900000.0, 'CBA': 1350000.0, 'CSL': 1000000.0, 'RIO': 1000000.0, 'WES': 750000.0}
{'CBA': 0.27, 'CSL': 0.2, 'RIO': 0.2}


## 9. Introduction to generator expressions

**Concept:** A generator expression produces values lazily instead of materialising the whole result. List comprehension uses [] where as generator uses ()

In [ ]:
class TradeStream:
    def __init__(self, trades): 
        self.trades = trades
    def notionals(self):
        return (t["quantity"]*t["price"] for t in self.trades)

stream = TradeStream([
    {"ticker":"BHP","quantity":2400,"price":41.70},
    {"ticker":"CBA","quantity":800,"price":178.25},
    {"ticker":"CSL","quantity":350,"price":312.40},
    {"ticker":"RIO","quantity":1200,"price":119.80},
])
for value in stream.notionals(): print(f"${value:,.0f}")

$100,080
$142,600
$109,340
$143,760


## 10. List comprehensions vs. generators

**Concept:** Lists are eager and reusable; generators are lazy and especially useful for large, one-pass datasets.

In [5]:
class Dataset:
    def __init__(self, prices): self.prices = prices
    def list_returns(self):
        return [self.prices[i]/self.prices[i-1]-1 for i in range(1,len(self.prices))]
    def generator_returns(self):
        return (self.prices[i]/self.prices[i-1]-1 for i in range(1,len(self.prices)))

d = Dataset([41.70,42.15,41.92,43.08,42.76,44.21])
print(type(d.list_returns()))
g = d.generator_returns()
print(type(g))
print(next(g))
print(list(g))

<class 'list'>
<class 'generator'>
0.010791366906474753
[-0.005456702253855195, 0.02767175572519065, -0.00742804085422466, 0.03391019644527593]


## 11. Write your own generator expressions

**Concept:** Assigning a generator expression to a variable lets a downstream process consume only the records it needs.

In [7]:
class OrderBook:
    def __init__(self, orders): self.orders = orders
    def large_orders(self, minimum):
        return (o for o in self.orders if o["quantity"]*o["price"] >= minimum)

book = OrderBook([
    {"ticker":"BHP","quantity":500,"price":41.70},
    {"ticker":"CBA","quantity":3000,"price":178.25},
    {"ticker":"CSL","quantity":150,"price":312.40},
    {"ticker":"RIO","quantity":2500,"price":119.80},
])
g = book.large_orders(200_000)
print(next(g))
print(list(g))

{'ticker': 'CBA', 'quantity': 3000, 'price': 178.25}
[{'ticker': 'RIO', 'quantity': 2500, 'price': 119.8}]


## 12. Changing the output in generator expressions

**Concept:** A generator can lazily transform each selected record into a new business object or dictionary.

In [8]:
class ExposureStream:
    def __init__(self, positions): self.positions = positions
    def records(self, portfolio_value):
        return ({"ticker":ticker,"exposure":weight*portfolio_value}
                for ticker,weight in self.positions.items())

positions = {"BHP":.18,"CBA":.27,"CSL":.20,"RIO":.20,"WES":.15}
for row in ExposureStream(positions).records(10_000_000):
    print(f"{row['ticker']}: ${row['exposure']:,.0f}")

BHP: $1,800,000
CBA: $2,700,000
CSL: $2,000,000
RIO: $2,000,000
WES: $1,500,000


## 13. Build a generator

**Concept:** A generator function uses `yield` to produce results incrementally while preserving its local state between outputs.

In [10]:
class PriceSignalGenerator:
    def __init__(self, prices, threshold): self.prices, self.threshold = prices, threshold
    def signals(self):
        previous = None
        for price in self.prices:
            if previous is None:
                previous = price
                continue
            change = price/previous - 1
            if abs(change) >= self.threshold:
                yield {"price":price,"change":change,"signal":"BUY" if change > 0 else "SELL"}
            previous = price

g = PriceSignalGenerator([41.70,41.85,42.60,42.30,43.50,43.45,42.70], .01)
for signal in g.signals(): print(signal)

{'price': 42.6, 'change': 0.017921146953405076, 'signal': 'BUY'}
{'price': 43.5, 'change': 0.028368794326241176, 'signal': 'BUY'}
{'price': 42.7, 'change': -0.017261219792865323, 'signal': 'SELL'}
